In [1]:
# import libs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# load data
df = pd.read_csv("../data/2_dataset.csv", low_memory=False)
# df that we will use for EDA
eda_df = df.copy()

# Exploratory Data Analysis (EDA) 

## Preface 

Previous treatments :

* 1_dataset_engineering.ipynb

The explanaition of all the variables of the dataset is done in the notebook : 1_dataset_engineering.ipynb

## Objective

Our goal here is to analyze the existing variables in relation to our target (delay in minutes of a bus) to determine what treatments we will do to our data to reveal the patterns that we found.

## Table of contents

## Dataset global overview

Before getting into the univariate analysis of each one of our dataset's variables, we will first observe the global overview of the dataset.

### Dataset format

In [2]:
df.shape

(50913, 17)

We have 50913 entries for 17 variables. As mentionned previously the details for each variable can be found in `1_dataset_engineering.ipynb`.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50913 entries, 0 to 50912
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ROUTE              50385 non-null  object 
 1   LOCAL_TIME         50913 non-null  object 
 2   WEEK_DAY           50913 non-null  object 
 3   DELAY              50913 non-null  int64  
 4   LOCAL_MONTH        50875 non-null  float64
 5   LOCAL_DAY          50875 non-null  float64
 6   TEMP               50866 non-null  float64
 7   DEW_POINT_TEMP     50866 non-null  float64
 8   HUMIDEX            10798 non-null  float64
 9   PRECIP_AMOUNT      50866 non-null  float64
 10  RELATIVE_HUMIDITY  50866 non-null  float64
 11  STATION_PRESSURE   50866 non-null  float64
 12  VISIBILITY         50856 non-null  float64
 13  WEATHER_ENG_DESC   7517 non-null   object 
 14  WINDCHILL          5388 non-null   float64
 15  WIND_DIRECTION     49228 non-null  float64
 16  WIND_SPEED         508

### Missing data

In [4]:
def missing_data(df):
    # Calculate missing values and proportions
    missing_count = df.isna().sum()
    missing_percent = (df.isna().mean() * 100).round(2)

    # Combine into one DataFrame
    missing_df = pd.DataFrame({
        'Missing Values': missing_count,
        'Percentage (%)': missing_percent
    })

    return missing_df

missing_data(eda_df)

,Missing Values,Percentage (%)
ROUTE,528,1.04
LOCAL_TIME,0,0.00
WEEK_DAY,0,0.00
DELAY,0,0.00
LOCAL_MONTH,38,0.07
LOCAL_DAY,38,0.07
TEMP,47,0.09
DEW_POINT_TEMP,47,0.09
HUMIDEX,40115,78.79
PRECIP_AMOUNT,47,0.09


#### Handling HUMIDEX and WINDCHILL

We notice that we have a lot of missing data in the following variables :

* `HUMIDEX` (78.79% of missing data)
* `WINDCHILL` (89.42% of missing data)

These are index calculated by the providers of the dataset. In their documentation, they give the formulas for both indexes :

* Humidex :
    * With :
        - $T_{\text{air}}$ : air temperature (in Celsius)
        - $T_{\text{dewpoint}}$ : dewpoint temperature (in Kelvin)
        - $e$ : vapour pressure (in hPa)

$$ humidex = T_{\text{air}} + h $$

$$ h = 0.5555 \times (e - 10.0) $$

$$ e = 6.11 \exp\left[ 5417.7530 \times \left( \frac{1}{273.15} - \frac{1}{T_{\text{dewpoint}}} \right) \right] $$

$$ \exp = 2.71828 $$

*  Windchill :
    * With :
        - $T_{\text{air}}$ : air temperature (in Celsius)
        - $V_{\text{10m}}$ : wind speed at 10 meters (in kilometers per hour)
    * For when $T_{\text{air}} \le 0 \text{°C}$ and $V_{\text{10m}} \ge 5 \text{km/h}$ 
        1. $windchill = 13.12 + 0.6215 T_{\text{air}} - 11.37 V_{\text{10m}}^{0.16} + 0.3965 T_{\text{air}} V_\text{10m}^{0.16} $
    * For when $T_{\text{air}} \le 0 \text{°C}$ and $0 < V_{\text{10m}} < 5 \text{km/h}$
        
        2. $windchill = T_{\text{air}} + ((-1.59 + 0.1345T_{\text{air}})/5) \times V_{\text{10m}}$

It happens that we have all the required data to calculate the humidex and windchill indexes. We can fill in those gaps by calculating these values manually. However, it is important to note that our value for the windspeed is *usually* measured at 10 meters and that this index may be calculated at a different altitude. Furthermore, not all entries will meet the requirements for calculating the winchill index, we still may have a high proportion of missing data.

> Note : In the original dataset, the humidex and windchill are rounded to the nearest degree. To increase precision we recalculate even the non-missing values. Furthermore, in the original dataset, the humidex was provided only when the air temperature was greater or equal to 20°C and the humidex value is at least 1 degree greater than the air temperature.

We replace the data with our formulas and look how much data is missing :

In [5]:
def humidex(t_air, t_dewpoint):
    exp = 2.71828 # we use the exponential value provided by the documentation, we could be more precise by using numpy.exp(1)
    t_dewpoint += 273.15 # celsius to kelvin
    e = 6.11 * exp**(5417.7530 * ((1/273.15) - (1/t_dewpoint)))  # in hPa
    h = 0.5555 * (e - 10.0)
    humidex = t_air + h
    return humidex

def wind_chill(t_air, windspeed):
    if t_air <= 0 and windspeed >= 5:
        return 13.12 + 0.6215 * t_air - 11.37 * (windspeed**0.16) + 0.3965 * t_air * (windspeed**0.16)
    if t_air <= 0 and 0 < windspeed and windspeed < 5:
        return t_air + ((-1.59 + 0.1345 * t_air) / 5) * windspeed
    else:
        return np.nan

# Recalculate HUMIDEX and WIND_CHILL
eda_df["HUMIDEX"] = eda_df.apply(lambda row: humidex(row['TEMP'], row['DEW_POINT_TEMP']), axis=1)
eda_df["WINDCHILL"] = eda_df.apply(lambda row: wind_chill(row['TEMP'], row['WIND_SPEED']), axis=1)

# Show the empty values after recalculation
missing_data(eda_df[['HUMIDEX', 'WINDCHILL']])

,Missing Values,Percentage (%)
HUMIDEX,47,0.09
WINDCHILL,45525,89.42


We notice that the the humidex now only has 0.09% of it's data missing while the windchill stays at 89.42%.

> [TODO] FIND OUT WHAT TO DO WITH WINDCHILL. FOR NOW NO ACTIONS TAKEN

With these two special cases taken care of, we can move on to the next one.

#### Handling the missing WEATHER_ENG_DESC values

Another special case is the one of the weather description. This variable has 85.24% of it's data missing. However, according to the source documentation, this is not due to a lack of data but rather a lack of special event. So no data means no special event. Hence, we will replace all the missing values by the value "Clear".

> Note: the value "Clear" can be missleading as it could be interpreted as a clear sky. Here it just means that there are no special event, not that the sky isn't cloudy. 


In [ ]:
# Replace missing WEATHER_ENG_DESC by 'Clear'
eda_df['WEATHER_ENG_DESC'] = eda_df['WEATHER_ENG_DESC'].fillna('Clear')
missing_data(eda_df[['WEATHER_ENG_DESC']])

,Missing Values,Percentage (%)
WEATHER_ENG_DESC,0,0.0


#### Dropping missing ROUTE values

Amongst the other variables two others stand out :

* `WIND_DIRECTION` (3.31% of missing data) 
* `ROUTE` (1.04% of missing data)

The missing data for the wind direction isn't a big problem and can be kept in the dataset. However, the data missing for the route is a bit more complex. We can't predict the bus delay without knowing what route is concerned. We should drop this variable and keep the other variables.

In [7]:
eda_df = eda_df.dropna(subset=['ROUTE'])
# Verify the result
print(f"Shape after dropping missing ROUTE values: {eda_df.shape} / previous shape: {df.shape} (dropped {df.shape[0] - eda_df.shape[0]} rows)")

Shape after dropping missing ROUTE values: (50385, 17) / previous shape: (50913, 17) (dropped 528 rows)


#### Removing rows with missing weather data

Another compelling point is the fact that 47 rows seem to be missing weather data (even after the recalculations, the humidex has 47 rows of missing data). We want to verify if these rows are actually missing their entire weather data. If so, we should remove these rows from our dataset 

In [8]:
# Get the columns that have 47 or less data missing 
missing_data_df = eda_df.loc[:, eda_df.isna().sum() <= 47].isna()

# Get rows that have at least one missing value
rows_with_missing = eda_df[missing_data_df.any(axis=1)]

print(f"Number of rows with missing values: {len(rows_with_missing)}")

Number of rows with missing values: 47


With this verification, we could have had more than 47 rows concerned by the missing data. But by having precisely 47 rows concerned, we can deduce that these 47 rows have a big amount of missing data. We are not interested by these rows. Being a very small amount of data relative to the entire dataset, we drop them. 

In [9]:
before_drop_shape = eda_df.shape
# Dropping the 47 rows with missing data
eda_df = eda_df.drop(rows_with_missing.index)
# Verify the result
print(f"Shape after dropping rows with missing data: {eda_df.shape} / previous shape: {before_drop_shape} (dropped {before_drop_shape[0] - eda_df.shape[0]} rows)")

Shape after dropping rows with missing data: (50338, 17) / previous shape: (50385, 17) (dropped 47 rows)
